In [ ]:
!pip -q install -U gdown
!gdown --fuzzy "https://drive.google.com/file/d/17mu3214-Wpq3e9iNKIn75N1pTUmBbdUY/view?usp=sharing" -O /content/archive.zip

In [ ]:
!unzip -q /content/archive.zip -d /content/data

In [ ]:
!apt-get -qq update
!apt-get -qq install -y ffmpeg


In [ ]:

!pip uninstall -y \
  whisperx pyannote.audio lightning torchmetrics \
  torch torchvision torchaudio torchcodec


In [ ]:
!pip install -U pip setuptools wheel

In [ ]:
!pip install --no-cache-dir "whisperx==3.8.1"


In [ ]:
!pip install --no-cache-dir --force-reinstall \
  "torchmetrics==1.6.1" \
  "lightning>=2.4,<2.6" \
  "pyannote.audio>=4.0.4,<4.1"

In [ ]:
!pip install --no-cache-dir --force-reinstall "huggingface-hub==0.36.0"
!pip check

In [ ]:
import importlib.metadata as md

print("huggingface-hub:", md.version("huggingface-hub"))
print("transformers:", md.version("transformers"))
print("torch:", md.version("torch"))
print("torchaudio:", md.version("torchaudio"))
print("whisperx:", md.version("whisperx"))

In [ ]:
import torch
import whisperx
import pyannote.audio

print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
print("whisperx ok")
print("pyannote ok")

In [ ]:
from __future__ import annotations

import gc
import json
import os
import re
from pathlib import Path
from typing import Dict, Tuple

import torch
import whisperx
from whisperx.diarize import DiarizationPipeline


# =========================================
# CONFIG
# =========================================
INPUT_ROOT = Path("/content/data/wav_only")   # interview folders 300_P, 301_P, ... live here
OUTPUT_DIR = Path("/content/transcribed")

HF_TOKEN = os.getenv("HF_TOKEN", "")  # set it with: %env HF_TOKEN=hf_xxx
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
COMPUTE_TYPE = "float16" if DEVICE == "cuda" else "int8"
BATCH_SIZE = 16 if DEVICE == "cuda" else 4
MODEL_NAME = "large-v3" if DEVICE == "cuda" else "medium"
LANGUAGE_CODE = "en"

MIN_SPEAKERS = 1
MAX_SPEAKERS = 6

# if True, skip when /content/transcribed/<id>.json already exists
SKIP_EXISTING = True

SKIP_IDS = {'345', '640', '334', '649', '316', '677', '322', '659', '381', '693', '311', '625', '350', '663', '641', '676', '708', '662', '607', '682', '307', '364', '327', '347', '330', '356', '366', '669', '336', '306', '695', '325', '304', '698', '602', '608', '667', '699', '688', '703', '331', '664', '352', '314', '655', '343', '369', '627', '359', '370', '666', '384', '338', '617', '374', '619', '638', '349', '373', '604', '651', '606', '679', '713', '303', '612', '603', '315', '344', '318', '379', '385', '341', '620', '635', '309', '335', '707', '380', '670', '339', '301', '673', '346', '631', '362', '337', '622', '609', '300', '320', '710', '697', '368', '333', '601', '340', '361', '375', '312', '329', '328', '689', '376', '715', '600', '382', '605', '383', '718', '687', '657', '323', '623', '302', '716', '624', '656', '633', '363', '717', '348', '326', '712', '305', '653', '629', '332', '354', '371', '377', '680', '321', '618', '654', '324', '705', '683', '308', '637', '692', '658', '652', '684', '691', '357', '342', '313', '628', '632', '650', '702', '367', '636', '365', '360', '358', '634', '319', '378', '310', '351', '615', '696', '660', '353', '317', '661', '355', '709', '372', '626'}
# =========================================
# HELPERS
# =========================================
def cleanup_memory() -> None:
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()


def extract_interview_id_from_folder_name(name: str) -> str:
    """
    From '300_P' -> '300'
    """
    m = re.match(r"^(\d+)_P$", name, flags=re.IGNORECASE)
    if not m:
        raise ValueError(f"Cannot extract interview id from folder name: {name}")
    return m.group(1)


def list_interview_dirs(root: Path) -> list[Path]:
    """
    Returns folders named <id>_P, sorted by id.
    """
    dirs = [
        p for p in root.iterdir()
        if p.is_dir() and re.match(r"^\d+_P$", p.name, flags=re.IGNORECASE)
    ]
    return sorted(dirs, key=lambda p: int(extract_interview_id_from_folder_name(p.name)))


def find_audio_file(interview_dir: Path) -> Path:
    """
    Looks for a *_AUDIO.wav file inside the interview folder.
    For example: /content/data/wav_only/300_P/300_AUDIO.wav
    """
    wavs = sorted(interview_dir.glob("*_AUDIO.wav"))
    if not wavs:
        wavs = sorted(interview_dir.rglob("*_AUDIO.wav"))
    if not wavs:
        raise FileNotFoundError(f"No *_AUDIO.wav found under {interview_dir}")
    return wavs[0]


def process_one_audio(
    wav_path: Path,
    asr_model,
    diarize_model,
    align_cache: Dict[str, Tuple[object, dict]],
) -> dict:
    audio = whisperx.load_audio(str(wav_path))

    result = asr_model.transcribe(
        audio,
        batch_size=BATCH_SIZE,
        language=LANGUAGE_CODE,
    )

    # Force the language to English
    result["language"] = LANGUAGE_CODE
    lang = LANGUAGE_CODE

    if lang not in align_cache:
        print(f"    loading align model for language={lang}")
        model_a, metadata = whisperx.load_align_model(
            language_code=lang,
            device=DEVICE
        )
        align_cache[lang] = (model_a, metadata)

    model_a, metadata = align_cache[lang]

    result = whisperx.align(
        result["segments"],
        model_a,
        metadata,
        audio,
        DEVICE,
        return_char_alignments=False,
    )

    diarize_segments = diarize_model(
        audio,
        min_speakers=MIN_SPEAKERS,
        max_speakers=MAX_SPEAKERS,
    )

    result = whisperx.assign_word_speakers(diarize_segments, result)

    del audio, diarize_segments
    cleanup_memory()

    return result


# =========================================
# MAIN
# =========================================
def main() -> None:
    if not HF_TOKEN:
        raise ValueError("HF_TOKEN is not set. Set it first via: %env HF_TOKEN=hf_xxx")

    if not INPUT_ROOT.exists():
        raise FileNotFoundError(f"INPUT_ROOT does not exist: {INPUT_ROOT}")

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    print(f"INPUT_ROOT={INPUT_ROOT}")
    print(f"OUTPUT_DIR={OUTPUT_DIR}")
    print(f"DEVICE={DEVICE}")
    print(f"COMPUTE_TYPE={COMPUTE_TYPE}")
    print(f"MODEL_NAME={MODEL_NAME}")
    print(f"LANGUAGE_CODE={LANGUAGE_CODE}")
    print(f"BATCH_SIZE={BATCH_SIZE}")
    print(f"SKIP_IDS count={len(SKIP_IDS)}")

    interview_dirs = list_interview_dirs(INPUT_ROOT)
    print(f"Found {len(interview_dirs)} interview folders")

    if not interview_dirs:
        return

    print("Loading WhisperX model once...")
    ASR_OPTIONS = {
        "temperatures": [0.0],   # keep decoding as deterministic as possible
        "beam_size": 5,          # keep 5 or reduce to 1
    }

    asr_model = whisperx.load_model(
        MODEL_NAME,
        DEVICE,
        compute_type=COMPUTE_TYPE,
        asr_options=ASR_OPTIONS,
    )

    print("Loading diarization pipeline once...")
    diarize_model = DiarizationPipeline(token=HF_TOKEN, device=DEVICE)

    align_cache: Dict[str, Tuple[object, dict]] = {}

    processed = 0
    failed = 0
    skipped_by_list = 0
    skipped_existing = 0

    for i, interview_dir in enumerate(interview_dirs, start=1):
        interview_id = extract_interview_id_from_folder_name(interview_dir.name)
        out_json = OUTPUT_DIR / f"{interview_id}.json"

        if interview_id in SKIP_IDS:
            print(f"[{i}/{len(interview_dirs)}] skip {interview_id}: in SKIP_IDS")
            skipped_by_list += 1
            continue

        if SKIP_EXISTING and out_json.exists():
            print(f"[{i}/{len(interview_dirs)}] skip {interview_id}: json already exists")
            skipped_existing += 1
            continue

        try:
            wav_path = find_audio_file(interview_dir)
            print(f"[{i}/{len(interview_dirs)}] transcribing {wav_path}")

            result = process_one_audio(
                wav_path=wav_path,
                asr_model=asr_model,
                diarize_model=diarize_model,
                align_cache=align_cache,
            )

            with open(out_json, "w", encoding="utf-8") as f:
                json.dump(result, f, ensure_ascii=False, indent=2)

            print(f"[{i}/{len(interview_dirs)}] saved {out_json}")
            processed += 1

            del result
            cleanup_memory()

        except Exception as e:
            failed += 1
            print(f"[{i}/{len(interview_dirs)}] ERROR for {interview_dir.name}: {e}")
            cleanup_memory()

    print(
        f"Done. processed={processed}, failed={failed}, "
        f"skipped_by_list={skipped_by_list}, skipped_existing={skipped_existing}"
    )



In [ ]:
main()